# 第11課 - 代理人對代理人 (A2A) 協議


## 設定


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## A2A 協議是甚麼？

**Agent-to-Agent (A2A) protocol** 是一個開放標準，讓 AI 代理能夠互相通訊、
發現彼此，並協作 — 即使它們建構於不同的框架或由不同的服務
託管。

Key concepts:

- **Discovery** – 代理會發佈一張 *Agent Card*，描述它們的能力，讓其他代理（或協調器）更容易找到適合處理任務的專家。
- **Message Passing** – 代理透過共同的協議交換結構化訊息，因此一個代理的請求可以被另一個代理理解並完成，無論其內部實作為何。
- **Task Lifecycle** – A2A 定義了狀態，例如 *submitted*, *working*, *completed*, 及
  *failed*, 讓協調器完全看見被委派任務的進度。

In this lesson we simulate A2A-style collaboration by wiring three specialized travel agents
串接到一個工作流程，讓每個代理貢獻其專長並把結果傳遞給下一個。


## 建立專門化的旅遊代理


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## 透過工作流程的多代理協作

我們把這三個代理連接成一個順序工作流程，以模仿 A2A 訊息傳遞：

1. **CurrencyExchangeAgent** 接收用戶請求並提供貨幣指引。
2. **ActivityPlannerAgent** 接收增強的上下文並加入活動建議。
3. **TravelManagerAgent** 將兩者輸入綜合成最終的旅行簡報。


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## 在生產環境中理解 A2A

在生產環境中，A2A 協議釋放了強大的跨服務情境：

| Capability | Description |
|---|---|
| **跨框架互通** | 用一個框架構建的代理可以將任務委派給使用任何其他符合 A2A 的框架構建的代理，實現真正的跨組織互通。 |
| **服務邊界** | 代理可以分佈在不同的微服務、雲端區域，甚至不同的組織中，仍能無縫協作。 |
| **動態發現** | 協調器可以在運行時查詢代理卡註冊表，以為特定子任務找到最合適的專家。 |
| **串流與推送通知** | A2A 支援 Server-Sent Events (SSE) 以提供即時進度更新，並為長時間運行的任務提供推送通知。 |

我們上面建立的工作流程是此模式的簡化、進程內版本。在真實部署中，每個代理會公開一個 HTTP 端點、發佈代理卡，並透過 A2A JSON-RPC 協議進行通訊。


## 摘要

在本課程中你學到：

1. **A2A 協議是甚麼** — 一種開放標準，用於代理到代理的發現、訊息傳遞、
   與任務管理。
2. **如何建立專門化的代理** — 一個貨幣兌換代理、 一個活動規劃代理、
   以及一個旅行管理協調器。
3. **如何將代理連接成工作流程** — 使用 `WorkflowBuilder` 來建模順序
   代理之間的訊息傳遞。
4. **A2A 在生產環境中的運作方式** — 使能跨框架、跨服務的協作
   透過動態發現與串流更新。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
免責聲明：
本文件已使用 AI 翻譯服務「Co-op Translator」（https://github.com/Azure/co-op-translator）進行翻譯。雖然我們力求準確，但請注意自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為具權威性的來源。對於重要資訊，建議採用專業人工翻譯。我們對因使用此翻譯而引致的任何誤解或誤釋概不負責。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
